# LangGraph Agent 

In [ ]:
%load_ext autoreload
%autoreload 2

#### Authenticate with AWS

In [ ]:
import os
os.environ['AWS_PROFILE'] = 'genaiic-awsi+eng-ravividy-agentic-platform-admin'
os.environ['UAEF_LLM_MODEL_ID']="us.anthropic.claude-sonnet-4-6"

In [ ]:
from dotenv import load_dotenv
loaded = load_dotenv()
print(loaded)

#### Import metrics for evaluation

In [ ]:
import metrics
from metrics import *


for name in dir(metrics):
    val = getattr(metrics, name)
    if isinstance(val, list):
        print(f"{name}:")
        for m in val:
            print(f"  - {m}")
        print()


#### Choose the set of metrics for evaluation

In [ ]:
metrics = single_ag_metrics
print("You've chosen the following metrics for evaluating the LangGraph agent:")
for m in metrics:
    print(f"  - {m}")

In [ ]:
from uaef.data import parse_ground_truth_row
# Import UAEF
from uaef.api import evaluate, batch_evaluate
from uaef.models import GroundTruth, ToolCall
from datetime import datetime
print("✓ UAEF imported successfully!")

## Single Agent: Booking Assistant

#### Step 1: Set up the database and booking tools

In [ ]:
import pandas as pd
import numpy as np
import os
import shutil
import uuid
import sqlite3
import boto3
import functools
import requests
import pytz
import time
import warnings
from ast import literal_eval

from IPython.display import Image, display
from botocore.config import Config
from typing import Annotated, Literal, Optional, Union
from typing_extensions import TypedDict
from datetime import date, datetime

# from langchain.globals import set_debug
from pydantic import BaseModel, Field
from langchain.tools import BaseTool, tool
from langchain_core.messages import AIMessage, BaseMessage, HumanMessage, ToolMessage, RemoveMessage
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables import RunnableConfig, RunnableLambda, Runnable
from langchain_aws import ChatBedrockConverse

from langgraph.graph import END, StateGraph, START
from langgraph.prebuilt import create_react_agent, ToolNode, tools_condition
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph.message import AnyMessage, add_messages

from uaef.adapters import LangGraphAdapter

# set_debug(False)
warnings.filterwarnings("ignore")

In [ ]:
db_url = "https://storage.googleapis.com/benchmarks-artifacts/travel-db/travel2.sqlite"
local_file = "travel2.sqlite"
# The backup lets us restart for each tutorial section
backup_file = "travel2.backup.sqlite"
overwrite = True
if overwrite or not os.path.exists(local_file):
    response = requests.get(db_url, timeout=30)
    response.raise_for_status()  # Ensure the request was successful
    with open(local_file, "wb") as f:
        f.write(response.content)
    # Backup - we will use this to "reset" our DB in each section
    shutil.copy(local_file, backup_file)


# Convert the flights to present time for our tutorial
def update_dates(file):
    shutil.copy(backup_file, file)
    conn = sqlite3.connect(file)
    cursor = conn.cursor()

    tables = pd.read_sql(
        "SELECT name FROM sqlite_master WHERE type='table';", conn
    ).name.tolist()
    tdf = {}
    # Table names come from sqlite_master (the DB's own schema); treat that
    # set as an allowlist and confirm each is a plain identifier before
    # interpolating, since SQLite cannot bind a table name as a parameter.
    allowed_tables = set(tables)
    for t in tables:
        if t not in allowed_tables or not t.isidentifier():
            raise ValueError(f"Refusing to query unexpected table name: {t!r}")
        tdf[t] = pd.read_sql(f'SELECT * from "{t}"', conn)

    example_time = pd.to_datetime(
        tdf["flights"]["actual_departure"].replace("\\N", pd.NaT)
    ).max()
    current_time = pd.to_datetime("now").tz_localize(example_time.tz)
    time_diff = current_time - example_time

    tdf["bookings"]["book_date"] = (
        pd.to_datetime(tdf["bookings"]["book_date"].replace("\\N", pd.NaT), utc=True)
        + time_diff
    )

    datetime_columns = [
        "scheduled_departure",
        "scheduled_arrival",
        "actual_departure",
        "actual_arrival",
    ]
    for column in datetime_columns:
        tdf["flights"][column] = (
            pd.to_datetime(tdf["flights"][column].replace("\\N", pd.NaT)) + time_diff
        )

    for table_name, df in tdf.items():
        df.to_sql(table_name, conn, if_exists="replace", index=False)
    del df
    del tdf
    conn.commit()
    conn.close()

    return file


def reset_db():
    """Reset db to a clean state before each evaluation run.
    In eval_mode: copies the pristine backup to local_file (no date shift).
    In production mode: copies backup then shifts dates to now.
    Always sets the module-level db to local_file so write tools never touch the backup.
    """
    global db
    if eval_mode:
        shutil.copy(backup_file, local_file)
    else:
        update_dates(local_file)
    db = local_file


db = update_dates(local_file)
# Capture the reference time from the raw backup so evaluation can freeze the DB at this point.
# eval_mode uses the backup directly (no date shift) and passes db_reference_time to the
# system prompt so the agent sees a consistent "current time" across runs.
_conn = sqlite3.connect(backup_file)
db_reference_time = pd.to_datetime(
    pd.read_sql("SELECT actual_departure FROM flights", _conn)["actual_departure"].replace("\\N", pd.NaT)
).max()
_conn.close()
print(f"✓ DB ready  |  db_reference_time = {db_reference_time}")

In [ ]:
# eval_mode=True: freeze the DB to the raw backup (no date shift) and pin the system-prompt
# time to db_reference_time so the agent sees the same world every run.
eval_mode = True

if eval_mode:
    eval_time = db_reference_time
else:
    eval_time = datetime.now()

reset_db()  # initialise db for this session


#### Step 2: Define tools and build the graph

In [ ]:
from langchain_core.tools.base import InjectedToolArg

@tool
def fetch_user_flight_information(config: Annotated[RunnableConfig, InjectedToolArg]) -> list[dict]:
    """Fetch all tickets for the user along with corresponding flight information and seat assignments."""
    configuration = config.get("configurable", {})
    passenger_id = configuration.get("passenger_id", None)
    if not passenger_id:
        raise ValueError("No passenger ID configured.")

    conn = sqlite3.connect(db)
    cursor = conn.cursor()
    query = """
    SELECT DISTINCT
        t.ticket_no, t.book_ref,
        f.flight_id, f.flight_no, f.departure_airport, f.arrival_airport, f.scheduled_departure, f.scheduled_arrival,
        bp.seat_no, tf.fare_conditions
    FROM 
        tickets t
        JOIN ticket_flights tf ON t.ticket_no = tf.ticket_no
        JOIN flights f ON tf.flight_id = f.flight_id
        LEFT JOIN boarding_passes bp ON bp.ticket_no = t.ticket_no AND bp.flight_id = f.flight_id
    WHERE 
        t.passenger_id = ?
    """
    cursor.execute(query, (passenger_id,))
    rows = cursor.fetchall()
    column_names = [column[0] for column in cursor.description]
    results = [dict(zip(column_names, row)) for row in rows]
    cursor.close()
    conn.close()
    return results


@tool
def search_flights(
    departure_airport: Optional[str] = None,
    arrival_airport: Optional[str] = None,
    start_time: Optional[date | datetime] = None,
    end_time: Optional[date | datetime] = None,
    limit: int = 20,
) -> list[dict]:
    """Search for flights based on departure airport, arrival airport, and departure time range."""
    conn = sqlite3.connect(db)
    cursor = conn.cursor()
    query = "SELECT * FROM flights WHERE 1 = 1"
    params = []
    if departure_airport:
        query += " AND departure_airport = ?"
        params.append(departure_airport)
    if arrival_airport:
        query += " AND arrival_airport = ?"
        params.append(arrival_airport)
    if start_time:
        query += " AND scheduled_departure >= ?"
        params.append(start_time)
    if end_time:
        query += " AND scheduled_departure <= ?"
        params.append(end_time)
    query += " LIMIT ?"
    params.append(limit)
    cursor.execute(query, params)
    rows = cursor.fetchall()
    column_names = [column[0] for column in cursor.description]
    results = [dict(zip(column_names, row)) for row in rows]
    cursor.close()
    conn.close()
    return results


@tool
def update_ticket_to_new_flight(
    ticket_no: str, new_flight_id: int, *, config: RunnableConfig
) -> str:
    """Update the user's ticket to a new valid flight."""
    configuration = config.get("configurable", {})
    passenger_id = configuration.get("passenger_id", None)
    if not passenger_id:
        raise ValueError("No passenger ID configured.")
    conn = sqlite3.connect(db)
    cursor = conn.cursor()
    cursor.execute(
        "SELECT departure_airport, arrival_airport, scheduled_departure FROM flights WHERE flight_id = ?",
        (new_flight_id,),
    )
    new_flight = cursor.fetchone()
    if not new_flight:
        cursor.close()
        conn.close()
        return "Invalid new flight ID provided."
    column_names = [column[0] for column in cursor.description]
    new_flight_dict = dict(zip(column_names, new_flight))
    current_time = configuration.get("current_time", datetime.now())
    if current_time.tzinfo is None:
        current_time = current_time.replace(tzinfo=pytz.timezone("Etc/GMT-3"))
    departure_time = datetime.strptime(
        new_flight_dict["scheduled_departure"], "%Y-%m-%d %H:%M:%S.%f%z"
    )
    time_until = (departure_time - current_time).total_seconds()
    if time_until < (3 * 3600):
        return f"Not permitted to reschedule to a flight that is less than 3 hours from the current time. Selected flight is at {departure_time}."
    cursor.execute(
        "SELECT flight_id FROM ticket_flights WHERE ticket_no = ?", (ticket_no,)
    )
    current_flight = cursor.fetchone()
    if not current_flight:
        cursor.close()
        conn.close()
        return "No existing ticket found for the given ticket number."
    cursor.execute(
        "SELECT * FROM tickets WHERE ticket_no = ? AND passenger_id = ?",
        (ticket_no, passenger_id),
    )
    current_ticket = cursor.fetchone()
    if not current_ticket:
        cursor.close()
        conn.close()
        return f"Current signed-in passenger with ID {passenger_id} not the owner of ticket {ticket_no}"
    cursor.execute(
        "UPDATE ticket_flights SET flight_id = ? WHERE ticket_no = ?",
        (new_flight_id, ticket_no),
    )
    conn.commit()
    cursor.close()
    conn.close()
    return "Ticket successfully updated to new flight."


@tool
def cancel_ticket(ticket_no: str, *, config: RunnableConfig) -> str:
    """Cancel the user's ticket and remove it from the database."""
    configuration = config.get("configurable", {})
    passenger_id = configuration.get("passenger_id", None)
    if not passenger_id:
        raise ValueError("No passenger ID configured.")
    conn = sqlite3.connect(db)
    cursor = conn.cursor()
    cursor.execute(
        "SELECT flight_id FROM ticket_flights WHERE ticket_no = ?", (ticket_no,)
    )
    existing_ticket = cursor.fetchone()
    if not existing_ticket:
        cursor.close()
        conn.close()
        return "No existing ticket found for the given ticket number."
    cursor.execute(
        "SELECT flight_id FROM tickets WHERE ticket_no = ? AND passenger_id = ?",
        (ticket_no, passenger_id),
    )
    current_ticket = cursor.fetchone()
    if not current_ticket:
        cursor.close()
        conn.close()
        return f"Current signed-in passenger with ID {passenger_id} not the owner of ticket {ticket_no}"
    cursor.execute("DELETE FROM ticket_flights WHERE ticket_no = ?", (ticket_no,))
    conn.commit()
    cursor.close()
    conn.close()
    return "Ticket successfully cancelled."


@tool
def search_car_rentals(
    location: Optional[str] = None,
    name: Optional[str] = None,
    price_tier: Optional[str] = None,
    start_date: Optional[Union[datetime, date]] = None,
    end_date: Optional[Union[datetime, date]] = None,
) -> list[dict]:
    """Search for car rentals based on location, name, price tier, start date, and end date."""
    conn = sqlite3.connect(db)
    cursor = conn.cursor()
    query = "SELECT * FROM car_rentals WHERE 1=1"
    params = []
    if location:
        query += " AND location LIKE ?"
        params.append(f"%{location}%")
    if name:
        query += " AND name LIKE ?"
        params.append(f"%{name}%")
    cursor.execute(query, params)
    results = cursor.fetchall()
    conn.close()
    return [dict(zip([column[0] for column in cursor.description], row)) for row in results]


@tool
def book_car_rental(rental_id: int) -> str:
    """Book a car rental by its ID."""
    conn = sqlite3.connect(db)
    cursor = conn.cursor()
    cursor.execute("UPDATE car_rentals SET booked = 1 WHERE id = ?", (rental_id,))
    conn.commit()
    if cursor.rowcount > 0:
        conn.close()
        return f"Car rental {rental_id} successfully booked."
    else:
        conn.close()
        return f"No car rental found with ID {rental_id}."


@tool
def update_car_rental(
    rental_id: int,
    start_date: Optional[Union[datetime, date]] = None,
    end_date: Optional[Union[datetime, date]] = None,
) -> str:
    """Update a car rental's start and end dates by its ID."""
    conn = sqlite3.connect(db)
    cursor = conn.cursor()
    if start_date:
        cursor.execute("UPDATE car_rentals SET start_date = ? WHERE id = ?", (start_date, rental_id))
    if end_date:
        cursor.execute("UPDATE car_rentals SET end_date = ? WHERE id = ?", (end_date, rental_id))
    conn.commit()
    if cursor.rowcount > 0:
        conn.close()
        return f"Car rental {rental_id} successfully updated."
    else:
        conn.close()
        return f"No car rental found with ID {rental_id}."


@tool
def cancel_car_rental(rental_id: int) -> str:
    """Cancel a car rental by its ID."""
    conn = sqlite3.connect(db)
    cursor = conn.cursor()
    cursor.execute("UPDATE car_rentals SET booked = 0 WHERE id = ?", (rental_id,))
    conn.commit()
    if cursor.rowcount > 0:
        conn.close()
        return f"Car rental {rental_id} successfully cancelled."
    else:
        conn.close()
        return f"No car rental found with ID {rental_id}."


@tool
def search_hotels(
    location: Optional[str] = None,
    name: Optional[str] = None,
    price_tier: Optional[str] = None,
    checkin_date: Optional[Union[datetime, date]] = None,
    checkout_date: Optional[Union[datetime, date]] = None,
) -> list[dict]:
    """Search for hotels based on location, name, price tier, check-in date, and check-out date."""
    conn = sqlite3.connect(db)
    cursor = conn.cursor()
    query = "SELECT * FROM hotels WHERE 1=1"
    params = []
    if location:
        query += " AND location LIKE ?"
        params.append(f"%{location}%")
    if name:
        query += " AND name LIKE ?"
        params.append(f"%{name}%")
    cursor.execute(query, params)
    results = cursor.fetchall()
    conn.close()
    return [dict(zip([column[0] for column in cursor.description], row)) for row in results]


@tool
def book_hotel(hotel_id: int) -> str:
    """Book a hotel by its ID."""
    conn = sqlite3.connect(db)
    cursor = conn.cursor()
    cursor.execute("UPDATE hotels SET booked = 1 WHERE id = ?", (hotel_id,))
    conn.commit()
    if cursor.rowcount > 0:
        conn.close()
        return f"Hotel {hotel_id} successfully booked."
    else:
        conn.close()
        return f"No hotel found with ID {hotel_id}."


@tool
def update_hotel(
    hotel_id: int,
    checkin_date: Optional[Union[datetime, date]] = None,
    checkout_date: Optional[Union[datetime, date]] = None,
) -> str:
    """Update a hotel's check-in and check-out dates by its ID."""
    conn = sqlite3.connect(db)
    cursor = conn.cursor()
    if checkin_date:
        cursor.execute("UPDATE hotels SET checkin_date = ? WHERE id = ?", (checkin_date, hotel_id))
    if checkout_date:
        cursor.execute("UPDATE hotels SET checkout_date = ? WHERE id = ?", (checkout_date, hotel_id))
    conn.commit()
    if cursor.rowcount > 0:
        conn.close()
        return f"Hotel {hotel_id} successfully updated."
    else:
        conn.close()
        return f"No hotel found with ID {hotel_id}."


@tool
def cancel_hotel(hotel_id: int) -> str:
    """Cancel a hotel by its ID."""
    conn = sqlite3.connect(db)
    cursor = conn.cursor()
    cursor.execute("UPDATE hotels SET booked = 0 WHERE id = ?", (hotel_id,))
    conn.commit()
    if cursor.rowcount > 0:
        conn.close()
        return f"Hotel {hotel_id} successfully cancelled."
    else:
        conn.close()
        return f"No hotel found with ID {hotel_id}."


@tool
def search_trip_recommendations(
    location: Optional[str] = None,
    name: Optional[str] = None,
    keywords: Optional[str] = None,
) -> list[dict]:
    """Search for trip recommendations based on location, name, and keywords."""
    conn = sqlite3.connect(db)
    cursor = conn.cursor()
    query = "SELECT * FROM trip_recommendations WHERE 1=1"
    params = []
    if location:
        query += " AND location LIKE ?"
        params.append(f"%{location}%")
    if name:
        query += " AND name LIKE ?"
        params.append(f"%{name}%")
    if keywords:
        keyword_list = keywords.split(",")
        keyword_conditions = " OR ".join(["keywords LIKE ?" for _ in keyword_list])
        query += f" AND ({keyword_conditions})"
        params.extend([f"%{keyword.strip()}%" for keyword in keyword_list])
    cursor.execute(query, params)
    results = cursor.fetchall()
    conn.close()
    return [dict(zip([column[0] for column in cursor.description], row)) for row in results]


@tool
def book_excursion(recommendation_id: int) -> str:
    """Book an excursion by its recommendation ID."""
    conn = sqlite3.connect(db)
    cursor = conn.cursor()
    cursor.execute(
        "UPDATE trip_recommendations SET booked = 1 WHERE id = ?", (recommendation_id,)
    )
    conn.commit()
    if cursor.rowcount > 0:
        conn.close()
        return f"Trip recommendation {recommendation_id} successfully booked."
    else:
        conn.close()
        return f"No trip recommendation found with ID {recommendation_id}."


@tool
def update_excursion(recommendation_id: int, details: str) -> str:
    """Update a trip recommendation's details by its ID."""
    conn = sqlite3.connect(db)
    cursor = conn.cursor()
    cursor.execute(
        "UPDATE trip_recommendations SET details = ? WHERE id = ?",
        (details, recommendation_id),
    )
    conn.commit()
    if cursor.rowcount > 0:
        conn.close()
        return f"Trip recommendation {recommendation_id} successfully updated."
    else:
        conn.close()
        return f"No trip recommendation found with ID {recommendation_id}."


@tool
def cancel_excursion(recommendation_id: int) -> str:
    """Cancel a trip recommendation by its ID."""
    conn = sqlite3.connect(db)
    cursor = conn.cursor()
    cursor.execute(
        "UPDATE trip_recommendations SET booked = 0 WHERE id = ?", (recommendation_id,)
    )
    conn.commit()
    if cursor.rowcount > 0:
        conn.close()
        return f"Trip recommendation {recommendation_id} successfully cancelled."
    else:
        conn.close()
        return f"No trip recommendation found with ID {recommendation_id}."


print("✓ All booking tools defined")

In [ ]:
from langgraph.prebuilt import ToolNode
from langchain_core.runnables import RunnableLambda

def handle_tool_error(state) -> dict:
    error = state.get("error")
    tool_calls = state["messages"][-1].tool_calls
    return {
        "messages": [
            ToolMessage(
                content=f"Error: {repr(error)}\n please fix your mistakes.",
                tool_call_id=tc["id"],
            )
            for tc in tool_calls
        ]
    }

def create_tool_node_with_fallback(tools: list) -> dict:
    return ToolNode(tools).with_fallbacks(
        [RunnableLambda(handle_tool_error)], exception_key="error"
    )

from typing import Annotated
from typing_extensions import TypedDict
from langgraph.graph.message import add_messages
from langchain_core.messages import AnyMessage

class State(TypedDict):
    messages: Annotated[list[AnyMessage], add_messages]
    user_info: str

from langchain_aws import ChatBedrockConverse
from langchain_core.prompts import ChatPromptTemplate
from langgraph.graph import StateGraph, START
from langgraph.prebuilt import ToolNode, tools_condition
from langgraph.checkpoint.memory import MemorySaver
from datetime import datetime

# Bedrock client
region_name = "us-east-1"
my_config = Config(
    region_name=region_name,
    signature_version="v4",
    retries={"max_attempts": 3, "mode": "standard"},
)
bedrock_runtime = boto3.client(service_name="bedrock-runtime", config=my_config)

bedrock_llm = ChatBedrockConverse(
    client=bedrock_runtime,
    model="us.anthropic.claude-sonnet-4-6",#"anthropic.claude-3-sonnet-20240229-v1:0",
    max_tokens=1024,
    temperature=0.0,
)


class Assistant:
    def __init__(self, runnable):
        self.runnable = runnable

    def __call__(self, state: State, config: RunnableConfig):
        while True:
            configuration = config.get("configurable", {})
            passenger_id = configuration.get("passenger_id", None)
            current_time = configuration.get("current_time", datetime.now())
            state = {**state, "user_info": passenger_id, "time": current_time}
            result = self.runnable.invoke(state)
            if not result.tool_calls and (
                not result.content
                or isinstance(result.content, list)
                and not result.content[0].get("text")
            ):
                messages = state["messages"] + [("user", "Respond with a real output.")]
                state = {**state, "messages": messages}
            else:
                break
        return {"messages": result}


primary_assistant_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are a helpful customer support assistant for Swiss Airlines. "
            " Use the provided tools to search for flights, company policies, and other information to assist the user's queries. "
            " When searching, be persistent. Expand your query bounds if the first search returns no results. "
            " If a search comes up empty, expand your search before giving up."
            "\n\nCurrent user:\n<User>\n{user_info}\n</User>"
            "\nCurrent time: {time}.",
        ),
        ("placeholder", "{messages}"),
    ]
)

part_1_tools = [
    fetch_user_flight_information,
    search_flights,
    update_ticket_to_new_flight,
    cancel_ticket,
    search_car_rentals,
    book_car_rental,
    update_car_rental,
    cancel_car_rental,
    search_hotels,
    book_hotel,
    update_hotel,
    cancel_hotel,
    search_trip_recommendations,
    book_excursion,
    update_excursion,
    cancel_excursion,
]
part_1_assistant_runnable = primary_assistant_prompt | bedrock_llm.bind_tools(part_1_tools)

builder = StateGraph(State)
builder.add_node("assistant", Assistant(part_1_assistant_runnable))
builder.add_node("tools", create_tool_node_with_fallback(part_1_tools))
builder.add_edge(START, "assistant")
builder.add_conditional_edges("assistant", tools_condition)
builder.add_edge("tools", "assistant")

memory = MemorySaver()
graph = builder.compile(checkpointer=memory)
print("\u2713 Booking assistant graph compiled")


#### Step 3: Run the booking assistant

In [ ]:
reset_db()  # restore clean DB state before this run

import pandas as pd
import time
import uuid
from langchain_core.messages import HumanMessage

# Load first question from ground truth
_df = pd.read_excel("data/ground-truth-booking.xlsx")
_row = _df.iloc[0]

user_input = str(_row["query"])
print(f"Question: {user_input}")

thread_id = str(uuid.uuid4())
config = {
    "configurable": {
        "passenger_id": "3442 587242",
        "thread_id": thread_id,
        "current_time": eval_time,
    }
}

events = []
t_start = time.time()
for event in graph.stream(
    {"messages": [HumanMessage(content=user_input)]},
    config,
    stream_mode="updates",
):
    events.append(event)
latency = time.time() - t_start

print(f"Latency: {latency:.2f}s")
print(f"Events: {len(events)}")


#### Step 4: Use LangGraph Adapter

In [ ]:
from langchain_core.messages import ToolMessage
from uaef.adapters import LangGraphAdapter
from datetime import timezone

# Prepend human message so the adapter extracts it as MessageRole.USER
events_with_human = [{"__human__": {"messages": [HumanMessage(content=user_input)]}}] + events
langgraph_result = {
    "stream_events": events_with_human,
    "session_id": "lg_booking_001",
    "latency": latency,
}

adapter = LangGraphAdapter()
agent_trace = adapter.transform_to_canonical(langgraph_result)

print(f"✓ Transformed LangGraph output to AgentTrace")
print(f"  Trace ID: {agent_trace.trace_id}")
print(f"  Messages: {len(agent_trace.messages)}")
print(f"  Tool Calls: {len(agent_trace.tool_calls)}")

# Get final response
final_messages = [e for e in events if "assistant" in e]
if final_messages:
    final_msg = final_messages[-1]["assistant"]["messages"]
    print(f"\nAgent Response: {final_msg.content}")


#### Step 5: Evaluate

In [ ]:
from uaef.api import evaluate
from uaef.data.ground_truth import build_ground_truth

_, ground_truth = build_ground_truth(_row, events=events)

result = evaluate(
    trace=agent_trace,
    ground_truth=ground_truth,
    metrics=metrics,
)

print(f"\n{'='*50}")
print("Booking Assistant - EVALUATION RESULTS")
print(f"Overall Score: {result.overall_score:.2f}")
print(f"Passed: {'\u2713 Yes' if result.passed else '\u2717 No'}")

print(f"\n{'='*50}")
print("METRIC SCORES BY DIMENSION")
for dimension in result.dimension_results:
    print(f"\n{dimension.dimension_name} (aggregate: {dimension.aggregate_score:.2f}):")
    for metric in dimension.metric_scores:
        if metric.score is None:
            print(f"  {metric.metric_name}: N/A")
            print(f"    {metric.reasoning}")
        else:
            print(f"  {metric.metric_name}: {metric.score:.2f}")


### Batch Evaluation
Load questions and ground truth from `data/ground-truth.xlsx`, run each query through the Booking Assistant, then evaluate all traces.

#### Load Ground Truth Q & A

In [ ]:
import pandas as pd
import json
from datetime import datetime, timezone
from uuid import uuid4

from uaef.api import evaluate, batch_evaluate
from uaef.models import GroundTruth, ToolCall, AgentTrace, Message
from uaef.models.message import MessageRole

excel_path = "data/ground-truth.xlsx"
df = pd.read_excel(excel_path)

gt_json = df.to_dict(orient="records")

print(f"✓ Converted {len(gt_json)} rows to JSON")
print(f"✓ Loaded {len(df)} ground truth q/a from {excel_path}")
print(f"  Columns: {list(df.columns)}")
df.head()


#### Reuse the Booking Assistant

In [ ]:
# Reuse the graph and adapter — no agent recreation
from uaef.data.ground_truth import build_ground_truth

adapter = LangGraphAdapter()
turn_payloads = []
ground_truths = []

for i, row in enumerate(gt_json):
    reset_db()
    query = parse_ground_truth_row(row, query_only=True)

    thread_id = str(uuid4())
    run_config = {
        "configurable": {
            "passenger_id": "3442 587242",
            "thread_id": thread_id,
            "current_time": eval_time,
        }
    }

    t_start = time.time()
    events = []
    for event in graph.stream(
        {"messages": [HumanMessage(content=query)]},
        run_config,
        stream_mode="updates",
    ):
        events.append(event)
    latency = time.time() - t_start

    _, gt = build_ground_truth(row, events=events)

    # Build a per-turn payload (one turn per session in this batch)
    events_with_human = [{"__human__": {"messages": [HumanMessage(content=query)]}}] + events
    turn_payloads.append({
        "session_id": f"batch_booking_{i}",
        "turn_id": 1,
        "stream_events": events_with_human,
        "latency": latency,
    })
    ground_truths.append(gt)

    label = f"'{query[:50]}...'" if len(query) > 50 else f"'{query}'"
    print(f"  [{i+1}/{len(gt_json)}] {label}")

# Single adapter call returns one session dict per session_id (1 turn each)
traces = adapter.transform_to_canonical(turn_payloads)

print(f"\n\u2713 Ran {len(traces)} queries through the Booking Assistant")


#### Run evaluation on the agent traces

In [ ]:
# Batch evaluate all traces
batch_results = batch_evaluate(
    traces=traces,
    ground_truths=ground_truths,
    metrics=metrics,
    max_workers=4,
)

print(f"{'='*50}")
print(f"BATCH RESULTS — BOOKING ASSISTANT")
print(f"{'='*50}")
for i, r in enumerate(batch_results):
    status = "\u2713" if r.passed else "\u2717"
    print(f"  {status} Test {i+1}: {r.overall_score:.2f}")

avg = sum(r.overall_score for r in batch_results) / len(batch_results)
pr = sum(1 for r in batch_results if r.passed) / len(batch_results) * 100
print(f"\nAverage Score: {avg:.2f}")
print(f"Pass Rate: {pr:.1f}%")


In [ ]:
# Uncomment to print evaluation details
for i, result in enumerate(batch_results):
    print(f"test {i}:")
    for dimension in result.dimension_results:
        print(f"\n{dimension.dimension_name} (aggregate score: {dimension.aggregate_score:.2f}):")
        for metric in dimension.metric_scores:
            if metric.score is None:
                print(f"  {metric.metric_name}: {metric.score}")
                print(metric.reasoning)
            else:
                print(f"  {metric.metric_name}: {metric.score:.2f}")
    print('-'*30)


#### Export batch evaluation results

In [ ]:
# Save evaluation results
from uaef.utils import save_metric_results

filepath = save_metric_results(batch_results, gt_json, prefix="booking_assistant_batch_results")


## Multi-Turn: Booking Assistant Conversation

Replay the full 6-turn conversation from `data/ground-truth-booking.xlsx` through the booking assistant using a single shared `thread_id` (so the agent retains memory across turns), then evaluate the complete conversation with multi-turn metrics.

#### Step 1: Load conversation turns from CSV

In [ ]:
import pandas as pd
import json

mt_xlsx_path = "data/ground-truth-booking.xlsx"
mt_df = pd.read_excel(mt_xlsx_path)

# Use the first session_id found
session_id = mt_df["session_id"].iloc[0]
turns = mt_df[mt_df["session_id"] == session_id].to_dict(orient="records")

print(f"Session: {session_id}")
print(f"Turns: {len(turns)}")

for i, t in enumerate(turns):
    print(f"  [{i+1}] {t['query']}")


#### Step 2: Run all turns through the agent with shared memory

In [ ]:
import time
from langchain_core.messages import HumanMessage, ToolMessage


# Single thread_id so MemorySaver retains context across all turns
thread_id = str(uuid.uuid4())
mt_config = {
    "configurable": {
        "passenger_id": "3442 587242",
        "thread_id": thread_id,
        "current_time": eval_time,
    }
}

all_events = []
per_turn_events = []  # one list of events per turn
turn_latencies = []

reset_db()  # restore clean DB state before batch run
for i, turn in enumerate(turns):
    query = str(turn["query"])
    t_start = time.time()
    turn_events = []
    for event in graph.stream(
        {"messages": [HumanMessage(content=query)]},
        mt_config,
        stream_mode="updates",
    ):
        turn_events.append(event)
        all_events.append(event)
    per_turn_events.append(turn_events)
    turn_latencies.append(time.time() - t_start)
    print(f"  [{i+1}/{len(turns)}] '{query[:60]}' ({turn_latencies[-1]:.2f}s)")

total_latency = sum(turn_latencies)
print(f"\n\u2713 Completed {len(turns)} turns in {total_latency:.2f}s")


#### Step 3: Transform to AgentTrace using LangGraph adapter

In [ ]:
from uaef.adapters import LangGraphAdapter

# Build a list of per-turn payloads for the adapter. Each payload includes the
# turn's stream_events (with the human message prepended so the adapter assigns
# USER → ASST order correctly) plus pre-recorded latency.
turn_payloads = []
for i, (turn, turn_events, turn_latency) in enumerate(zip(turns, per_turn_events, turn_latencies)):
    turn_stream = [{"__human__": {"messages": [HumanMessage(content=str(turn["query"]))]}}] + turn_events
    turn_payloads.append({
        "session_id": session_id,
        "turn_id": i + 1,
        "stream_events": turn_stream,
        "latency": turn_latency,
    })

adapter = LangGraphAdapter()
sessions = adapter.transform_to_canonical(turn_payloads)
mt_session = sessions[0]  # single session in this cell

print(f"\u2713 Session adapted: {mt_session['session_id']}")
print(f"  Per-turn traces : {len(mt_session['per_turn_traces'])}")
print(f"  Full trace ID   : {mt_session['full_trace'].trace_id}")
print(f"  Full messages   : {len(mt_session['full_trace'].messages)}")
print(f"  Full tool calls : {len(mt_session['full_trace'].tool_calls)}")


#### Step 4: Evaluate 

In [ ]:
from uaef.api import evaluate
from uaef.data.ground_truth import build_multi_turn_ground_truth

mt_ground_truth = build_multi_turn_ground_truth(turns, events=all_events)

mt_result = evaluate(
    trace=mt_session,
    ground_truth=mt_ground_truth,
    metrics=metrics,
)

print(f"\n{'='*50}")
print("MULTI-TURN EVALUATION RESULTS")
print(f"Overall Score : {mt_result.overall_score:.2f}")
print(f"Passed        : {'\u2713 Yes' if mt_result.passed else '\u2717 No'}")
if mt_result.failures:
    print(f"Failures      : {mt_result.failures}")

print(f"\n{'='*50}")
print("METRIC SCORES BY DIMENSION")
for dimension in mt_result.dimension_results:
    print(f"\n{dimension.dimension_name} (aggregate: {dimension.aggregate_score:.2f}):")
    for metric in dimension.metric_scores:
        score_str = f"{metric.score:.2f}" if metric.score is not None else "N/A"
        print(f"  {metric.metric_name}: {score_str}")
        if metric.score is None:
            print(f"    {metric.reasoning[:120]}")

# Per-turn metric scores
per_turn = mt_result.metadata.get("per_turn")
if per_turn:
    averages = mt_result.metadata.get("per_turn_averages", {})
    metric_names = list(per_turn[0]["metric_scores"].keys())
    col_w = 24
    print(f"\n{'='*65}")
    print("PER-TURN METRIC SCORES")
    header = f"  {'Metric':<{col_w}}" + "".join(f"{'Turn ' + str(t['turn']):>8}" for t in per_turn) + f"{'Average':>9}"
    print(header)
    print("  " + "-" * (len(header) - 2))
    for mname in metric_names:
        row = f"  {mname:<{col_w}}"
        for t in per_turn:
            sc = t["metric_scores"].get(mname)
            row += f"{sc:>8.2f}" if sc is not None else f"{'N/A':>8}"
        avg = averages.get(mname)
        row += f"{avg:>9.2f}" if avg is not None else f"{'N/A':>9}"
        print(row)




### Batch Multi-Turn Evaluation

Load `data/ground-truth-booking.xlsx`, group rows by `session_id`, run each session as a connected multi-turn conversation (shared `thread_id` per session so MemorySaver retains context), then evaluate each session with `evaluate()` and report results side-by-side.

#### Step 1: Load and group by session_id

In [ ]:
import pandas as pd
import json

mt_batch_path = "data/ground-truth-booking.xlsx"
mt_batch_df = pd.read_excel(mt_batch_path)

# Group rows by session_id, preserving row order within each session
sessions = {}
for _, row in mt_batch_df.iterrows():
    sid = str(row["session_id"])
    sessions.setdefault(sid, []).append(row.to_dict())

print(f"✓ Loaded {len(mt_batch_df)} rows → {len(sessions)} sessions from {mt_batch_path}")
for sid, turns in sessions.items():
    queries = [t["query"][:55] for t in turns]
    print(f"  [{sid}]  {len(turns)} turns")
    for q in queries:
        print(f"      - {q!r}")


#### Step 2: Run each session through the agent with shared memory

In [ ]:
import time, uuid
from langchain_core.messages import HumanMessage
from uaef.adapters import LangGraphAdapter
from uaef.data.ground_truth import build_multi_turn_ground_truth

mt_batch_adapter = LangGraphAdapter()
all_turn_payloads = []   # flat list across all sessions
mt_batch_gts      = []
mt_batch_session_ids = []

for sid, turns in sessions.items():
    reset_db()
    print(f"\n\u2500\u2500 Session {sid} ({len(turns)} turns) \u2500\u2500")
    thread_id = str(uuid.uuid4())
    session_config = {
        "configurable": {
            "passenger_id": "3442 587242",
            "thread_id": thread_id,
            "current_time": eval_time,
        }
    }

    all_events = []

    for i, turn in enumerate(turns):
        query = str(turn["query"])
        t_start = time.time()
        turn_events = []
        for event in graph.stream(
            {"messages": [HumanMessage(content=query)]},
            session_config,
            stream_mode="updates",
        ):
            turn_events.append(event)
            all_events.append(event)
        turn_lat = time.time() - t_start
        print(f"  [{i+1}/{len(turns)}] {query[:60]!r}  ({turn_lat:.1f}s)")

        # Per-turn payload with human message prepended
        turn_stream = [{"__human__": {"messages": [HumanMessage(content=query)]}}] + turn_events
        all_turn_payloads.append({
            "session_id": sid,
            "turn_id": i + 1,
            "stream_events": turn_stream,
            "latency": turn_lat,
        })

    mt_batch_gts.append(build_multi_turn_ground_truth(turns, events=all_events))
    mt_batch_session_ids.append(sid)

# Single adapter call returns one session dict per unique session_id
mt_batch_traces = mt_batch_adapter.transform_to_canonical(all_turn_payloads)

print(f"\n\u2713 Collected {len(mt_batch_traces)} session traces")


#### Step 3: Evaluate each session and compare

In [ ]:
from uaef.api import batch_evaluate

mt_batch_results = batch_evaluate(
    traces=mt_batch_traces,
    ground_truths=mt_batch_gts,
    metrics=metrics,
    max_workers=4,
)

print(f"{'='*65}")
print(f"{'SESSION':<14} {'SCORE':>6}  {'PASS':<5}  DIMENSION SCORES")
print(f"{'='*65}")
for sid, result in zip(mt_batch_session_ids, mt_batch_results):
    dim_scores = "  ".join(
        f"{d.dimension_name[:8]}={d.aggregate_score:.2f}"
        for d in result.dimension_results
    )
    status = "\u2713" if result.passed else "\u2717"
    print(f"{sid:<14} {result.overall_score:>6.2f}  {status:<5}  {dim_scores}")

print(f"{'='*65}")

# Per-turn metric scores per session
col_w = 24
for sid, result in zip(mt_batch_session_ids, mt_batch_results):
    per_turn = result.metadata.get("per_turn")
    if not per_turn:
        continue
    averages = result.metadata.get("per_turn_averages", {})
    metric_names = list(per_turn[0]["metric_scores"].keys()) if per_turn else []
    print(f"\n{'='*65}")
    print(f"PER-TURN METRIC SCORES \u2014 {sid}")
    header = f"  {'Metric':<{col_w}}" + "".join(f"{'Turn ' + str(t['turn']):>8}" for t in per_turn) + f"{'Average':>9}"
    print(header)
    print("  " + "-" * (len(header) - 2))
    for mname in metric_names:
        row = f"  {mname:<{col_w}}"
        for t in per_turn:
            sc = t["metric_scores"].get(mname)
            row += f"{sc:>8.2f}" if sc is not None else f"{'N/A':>8}"
        avg = averages.get(mname)
        row += f"{avg:>9.2f}" if avg is not None else f"{'N/A':>9}"
        print(row)


In [ ]:
# Per-metric breakdown per session
for sid, result in zip(mt_batch_session_ids, mt_batch_results):
    print(f"\n── {sid} (overall={result.overall_score:.2f}, passed={result.passed}) ──")
    for dim in result.dimension_results:
        print(f"  {dim.dimension_name} ({dim.aggregate_score:.2f}):")
        for m in dim.metric_scores:
            score_str = f"{m.score:.2f}" if m.score is not None else "N/A"
            print(f"    {m.metric_name}: {score_str}")


## Multi-Agent LangGraph Example

This example builds a multi-agent system with a **researcher** and a **writer** agent.
The researcher gathers information, then hands off to the writer to produce the final output.
We use the LangGraph adapter's multi-agent support to produce a `MultiAgentTrace` and evaluate
with multi-agent metrics.

#### Step 1: Build a multi-agent LangGraph

In [ ]:
from typing import Annotated
from typing_extensions import TypedDict
from langchain_core.messages import HumanMessage
from langchain_core.tools import tool
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode
from langchain_aws import ChatBedrockConverse
import boto3
from botocore.config import Config

# Tools for the researcher agent
@tool
def search_web(query: str) -> str:
    """Search the web for information."""
    return f"Search results for '{query}': Python was created by Guido van Rossum in 1991. It emphasizes readability and simplicity."

@tool
def get_statistics(topic: str) -> str:
    """Get statistics about a topic."""
    return f"Statistics for '{topic}': Used by 48.2% of developers (Stack Overflow 2024). Over 400k packages on PyPI."

# Shared state
class MultiAgentState(TypedDict):
    messages: Annotated[list, add_messages]

# Setup Bedrock LLM
region_name = "us-east-1"
my_config = Config(
    region_name=region_name,
    signature_version='v4',
    retries={'max_attempts': 3, 'mode': 'standard'}
)
bedrock_runtime = boto3.client(service_name="bedrock-runtime", config=my_config)

# Researcher LLM (with tools)
researcher_llm = ChatBedrockConverse(
    client=bedrock_runtime,
    model_id="anthropic.claude-3-sonnet-20240229-v1:0",
    max_tokens=1024,
    temperature=0.0,
).bind_tools([search_web, get_statistics])

# Writer LLM (no tools — we clean tool blocks from message history before invoking)
writer_llm = ChatBedrockConverse(
    client=bedrock_runtime,
    model_id="anthropic.claude-3-sonnet-20240229-v1:0",
    max_tokens=1024,
    temperature=0.0,
)

# Agent nodes
def researcher_node(state: MultiAgentState):
    response = researcher_llm.invoke(state["messages"])
    return {"messages": [response]}

def writer_node(state: MultiAgentState):
    # Filter out tool_use and tool_result messages to avoid Bedrock validation error
    from langchain_core.messages import HumanMessage as HM, AIMessage as AIM
    clean_messages = []
    for msg in state['messages']:
        if isinstance(msg, AIM):
            # Strip tool_calls from AI messages, keep only text content
            if msg.content:
                clean_messages.append(AIM(content=msg.content))
        elif hasattr(msg, 'type') and msg.type == 'tool':
            # Convert tool results to plain human messages so writer sees the data
            clean_messages.append(HM(content=f"Tool result: {msg.content}"))
        else:
            clean_messages.append(msg)
    clean_messages.append(HM(content="Based on the research above, write a concise 2-3 sentence summary."))
    response = writer_llm.invoke(clean_messages)
    return {"messages": [response]}

# Routing: if researcher wants to call tools, go to tools; otherwise go to writer
def route_researcher(state: MultiAgentState):
    last_msg = state["messages"][-1]
    if hasattr(last_msg, "tool_calls") and last_msg.tool_calls:
        return "tools"
    return "writer"

# Build the multi-agent graph
tools_list = [search_web, get_statistics]
graph_builder = StateGraph(MultiAgentState)
graph_builder.add_node("researcher", researcher_node)
graph_builder.add_node("tools", ToolNode(tools_list))
graph_builder.add_node("writer", writer_node)

graph_builder.add_edge(START, "researcher")
graph_builder.add_conditional_edges("researcher", route_researcher)
graph_builder.add_edge("tools", "researcher")  # tools feed back to researcher
graph_builder.add_edge("writer", END)

multi_agent_graph = graph_builder.compile()
print("\u2713 Multi-agent graph built: researcher -> tools -> researcher -> writer")

#### Step 2: Run the multi-agent graph

In [ ]:
import time

user_input = "Research Python programming language and write a brief summary."
ma_events = []
t_start = time.time()
for event in multi_agent_graph.stream(
    {"messages": [HumanMessage(content=user_input)]},
    stream_mode="updates"
):
    ma_events.append(event)
    for node_name in event:
        print(f"  [{node_name}] fired")
latency = time.time() - t_start

print(f"\nTotal events: {len(ma_events)}")
print(f"Latency: {latency:.2f}s")


In [ ]:
# Uncomment to save raw agent events
# import json

# all_events = []
# for i, event in enumerate(ma_events):
#     for node, output in event.items():
#         for msg in output.get("messages", []):
#             all_events.append({"event": i+1, "node": node, "message": msg.model_dump(mode="json")})

# with open("ma_events.json", "w") as f:
#     json.dump(all_events, f, indent=2, default=str)

# print(f"Dumped {len(all_events)} messages to ma_events.json")


#### Step 3: Transform to MultiAgentTrace using the adapter

In [ ]:

from uaef.adapters import LangGraphAdapter
from uaef.models.message import Message, MessageRole
from datetime import datetime, timezone

adapter = LangGraphAdapter()

# agent_node_names is inferred automatically from ma_events
multi_agent_trace = adapter.transform_to_canonical({
    "stream_events": [{"__human__": {"messages": [HumanMessage(content=user_input)]}}] + ma_events,
    "session_id": "ma_001",
    "latency": latency,
})

# Accumulate token counts from all agent messages across events
input_tokens, output_tokens = 0, 0
for event in ma_events:
    for node_output in event.values():
        if isinstance(node_output, dict):
            for msg in node_output.get("messages", []):
                usage = getattr(msg, "usage_metadata", {}) or {}
                input_tokens += usage.get("input_tokens", 0)
                output_tokens += usage.get("output_tokens", 0)
if input_tokens:
    multi_agent_trace.agent_traces["researcher"].input_tokens = input_tokens
if output_tokens:
    multi_agent_trace.agent_traces["researcher"].output_tokens = output_tokens

print(f"Trace type: {type(multi_agent_trace).__name__}")
print(f"Agents: {list(multi_agent_trace.agent_traces.keys())}")
print(f"Coordination events: {len(multi_agent_trace.coordination_events)}")
for agent_id, trace in multi_agent_trace.agent_traces.items():
    print(f"  {agent_id}: {len(trace.messages)} messages, {len(trace.tool_calls)} tool calls")
for event in multi_agent_trace.coordination_events:
    print(f"  {event.from_agent} -> {event.to_agent}: {event.event_type}")
print(f"  input_tokens={input_tokens}, output_tokens={output_tokens}, latency={latency:.2f}s")


#### Step 4: Evaluate with multi-agent metrics

In [ ]:
from uaef.api import evaluate
from uaef.models import GroundTruth, ToolCall
from datetime import datetime, timezone

ground_truth = GroundTruth(
    expected_output="Python was created by Guido van Rossum in 1991. It is used by 48.2% of developers according to Stack Overflow 2024.",
    expected_tool_calls=[
        ToolCall(
            name="search_web",
            arguments={"query": "Python programming language"},
            timestamp=datetime.now(timezone.utc)
        ),
        ToolCall(
            name="get_statistics",
            arguments={"topic": "Python"},
            timestamp=datetime.now(timezone.utc)
        ),
    ]
)

# evaluate() auto-detects MultiAgentTrace and uses MultiAgentEvaluator
result = evaluate(
    trace=multi_agent_trace,
    ground_truth=ground_truth,
    metrics=[
        "agent_utilization",
        "delegation_quality",
        "workflow_completion",
        "coordination_efficiency",
    ]
)

print(f"Overall Score: {result.overall_score:.2f}")
print(f"Passed: {result.passed}")
print(f"\n{'='*50}")
print("MULTI-AGENT METRIC SCORES")
for dimension in result.dimension_results:
    print(f"\n{dimension.dimension_name} (aggregate: {dimension.aggregate_score:.2f}):")
    for metric in dimension.metric_scores:
        score_str = f"{metric.score:.2f}" if metric.score is not None else "None"
        print(f"  {metric.metric_name}: {score_str}")
        print(f"    {metric.reasoning[:120]}")

#### Step 5: Evaluate with all metrics

In [ ]:
# Evaluate with all available metrics (auto-selects MultiAgentEvaluator)
# ground_truth with expected_tool_calls already defined in Step 4
result_all = evaluate(
    trace=multi_agent_trace,
    ground_truth=ground_truth,
    metrics=metrics,
    context=["Python was created by Guido van Rossum in 1991.",
             "Python is used by 48.2% of developers according to Stack Overflow 2024."]
)

print(f"Overall Score: {result_all.overall_score:.2f}")
print(f"Passed: {result_all.passed}")
if result_all.failures:
    print(f"Failures: {result_all.failures}")

print(f"\n{'='*60}")
print("ALL METRIC SCORES BY DIMENSION")
for dimension in result_all.dimension_results:
    valid = [m for m in dimension.metric_scores if m.score is not None]
    skipped = len(dimension.metric_scores) - len(valid)
    print(f"\n{dimension.dimension_name} (aggregate: {dimension.aggregate_score:.2f}, {skipped} skipped):")
    for metric in dimension.metric_scores:
        score_str = f"{metric.score:.2f}" if metric.score is not None else "N/A"
        print(f"  {metric.metric_name}: {score_str}")
        if metric.score is None:
            print(f"    reason: {metric.reasoning[:100]}")

#### Step 6: Batch evaluation with multi-agent

In [ ]:
import pandas as pd
import json
import time
from datetime import datetime, timezone
from uaef.api import batch_evaluate
from uaef.adapters import LangGraphAdapter
from uaef.models import GroundTruth, ToolCall
from langchain_core.messages import HumanMessage

excel_path = "data/ground-truth.xlsx"
df = pd.read_excel(excel_path)
gt_json = df.to_dict(orient="records")
print(f"✓ Loaded {len(gt_json)} ground truth entries")

adapter = LangGraphAdapter()
ma_traces = []
ma_ground_truths = []

for i, row in enumerate(gt_json):
    query = str(row.get("query", row.get("input", row.get("Question", ""))))
    expected = str(row.get("expected_output", row.get("expected", row.get("Answer", ""))))
    context = str(row.get("context", "")) if pd.notna(row.get("context")) else ""

    # Parse expected tool calls
    expected_tools = []
    raw_tools = row.get("expected_tool_calls", row.get("tools", None))
    if pd.notna(raw_tools) and raw_tools:
        try:
            parsed = json.loads(str(raw_tools)) if isinstance(raw_tools, str) else raw_tools
            if isinstance(parsed, list):
                for t in parsed:
                    expected_tools.append(ToolCall(
                        name=t.get("name", t.get("tool_name", "")),
                        arguments=t.get("arguments", t.get("parameters", {})),
                        timestamp=datetime.now(timezone.utc),
                    ))
        except (json.JSONDecodeError, TypeError):
            pass

    # Run the multi-agent graph
    t_start = time.time()
    events = []
    for event in multi_agent_graph.stream(
        {"messages": [HumanMessage(content=query)]},
        stream_mode="updates",
    ):
        events.append(event)
    latency = time.time() - t_start

    events_with_human = [{"__human__": {"messages": [HumanMessage(content=query)]}}] + events
    trace = adapter.transform_to_canonical({
        "stream_events": events_with_human,
        "session_id": f"ma_batch_{i}",
        "latency": latency,
    })
    ma_traces.append(trace)
    ma_ground_truths.append(GroundTruth(
        expected_output=expected,
        expected_tool_calls=expected_tools,
        context_documents=[context] if context else [],
    ))

    label = f"'{query[:50]}...'" if len(query) > 50 else f"'{query}'"
    trace_type = type(trace).__name__
    print(f"  [{i+1}/{len(gt_json)}] {label}  ({trace_type})")

print(f"\n✓ Ran {len(ma_traces)} queries through the multi-agent graph")

# Batch evaluate — MultiAgentEvaluator is auto-selected for MultiAgentTrace
ma_batch_results = batch_evaluate(
    traces=ma_traces,
    ground_truths=ma_ground_truths,
    metrics=metrics,
    max_workers=4,
)

print(f"\n{'='*50}")
print("BATCH RESULTS — MULTI-AGENT LANGGRAPH")
print(f"{'='*50}")
for i, r in enumerate(ma_batch_results):
    status = "✓" if r.passed else "✗"
    print(f"  {status} Test {i+1}: {r.overall_score:.2f}")

avg = sum(r.overall_score for r in ma_batch_results) / len(ma_batch_results)
pr = sum(1 for r in ma_batch_results if r.passed) / len(ma_batch_results) * 100
print(f"\nAverage Score: {avg:.2f}")
print(f"Pass Rate: {pr:.1f}%")

# Show batch-level metrics if available (e.g. agent_utilization_batch)
if ma_batch_results and "batch_metrics" in ma_batch_results[0].metadata:
    print(f"\n{'='*50}")
    print("BATCH-LEVEL MULTI-AGENT METRICS")
    for name, data in ma_batch_results[0].metadata["batch_metrics"].items():
        score_str = f"{data['score']:.2f}" if data['score'] is not None else "N/A"
        print(f"  {name}: {score_str}")

In [ ]:
# Uncomment to print evaluation details
for i, result in enumerate(ma_batch_results):
    print(f"test {i}:")
    for dimension in result.dimension_results:
        print(f"\n{dimension.dimension_name} (aggregate score: {dimension.aggregate_score:.2f}):")
        for metric in dimension.metric_scores:
            if metric.score is None:
                print(f"  {metric.metric_name}: {metric.score}")
                print(metric.reasoning)
            else:    
                print(f"  {metric.metric_name}: {metric.score:.2f}")
            
    print('-'*30)

## Multi-Turn: Multi-Agent Conversation

Run a multi-turn conversation through the researcher / writer multi-agent graph using a shared `thread_id` per session (so `MemorySaver` retains context across turns), then evaluate.

The flow mirrors the booking-agent multi-turn example exactly. Because the adapter now produces `MultiAgentTrace` per-turn traces (and a full `MultiAgentTrace`) when the graph has more than one agent node, multi-turn metrics (`context_retention`, `coherence`, ...) and multi-agent metrics (`agent_utilization`, `delegation_quality`, ...) are scored together in a single `evaluate()` call.

#### Step 1: Recompile the multi-agent graph with a checkpointer

The earlier multi-agent graph was compiled without a checkpointer, so it can't share state across `stream()` calls. Recompile from the same `graph_builder` with a `MemorySaver` so each session can use a shared `thread_id` to keep context across turns.

In [ ]:
from langgraph.checkpoint.memory import MemorySaver

# Recompile the multi-agent graph with a checkpointer so MemorySaver persists state across turns
ma_memory = MemorySaver()
multi_agent_graph_mt = graph_builder.compile(checkpointer=ma_memory)
print("✓ Multi-agent graph recompiled with MemorySaver")

#### Step 2: Define multi-turn conversations

In [ ]:
# Multi-turn conversations for the researcher / writer multi-agent system.
# Each row mirrors the schema parse_ground_truth_row() understands:
#   - 'query' / 'question'      — user input for that turn
#   - 'expected_output'         — expected assistant response
#   - 'expected_tool_calls'     — JSON list of {name, arguments} the agent should call
# build_multi_turn_ground_truth() pulls these per-turn fields and slots
# expected tool calls into ground_truth.expected_arguments['per_turn_tools'],
# which evaluate_multi_turn() then uses to score tool-calling metrics on each turn.
ma_mt_sessions = [
    {
        "session_id": "ma_mt_001",
        "description": "Research and summarize Python",
        "turns": [
            {
                "turn_id": 1,
                "query": "Research the Python programming language.",
                "expected_output": "Python was created by Guido van Rossum in 1991.",
                "expected_tool_calls": [
                    {"name": "search_web", "arguments": {"query": "Python programming language"}},
                ],
            },
            {
                "turn_id": 2,
                "query": "What are its main use cases?",
                "expected_output": (
                    "Python is used widely in web development, data science, machine learning, "
                    "automation, and scripting."
                ),
                "expected_tool_calls": [
                    {"name": "get_statistics", "arguments": {"topic": "Python use cases"}},
                ],
            },
            {
                "turn_id": 3,
                "query": "Now write a 3-sentence summary based on what you researched.",
                "expected_output": (
                    "Python was created by Guido van Rossum in 1991. It is used by 48.2% of "
                    "developers and has over 400k packages on PyPI. It is widely applied across "
                    "web development, data science, and automation."
                ),
                "expected_tool_calls": [],  # writer turn — no tools expected
            },
        ],
    },
    {
        "session_id": "ma_mt_002",
        "description": "Research and summarize JavaScript",
        "turns": [
            {
                "turn_id": 1,
                "query": "Research JavaScript.",
                "expected_output": "JavaScript is the scripting language of the web.",
                "expected_tool_calls": [
                    {"name": "search_web", "arguments": {"query": "JavaScript programming language"}},
                    {"name": "get_statistics", "arguments": {"topic": "JavaScript"}},
                ],
            },
            {
                "turn_id": 2,
                "query": "Summarize the key facts in 2-3 sentences.",
                "expected_output": (
                    "JavaScript is a widely-used programming language. It powers most modern "
                    "web applications."
                ),
                "expected_tool_calls": [],
            },
        ],
    },
]

print(f"\u2713 Defined {len(ma_mt_sessions)} multi-turn multi-agent sessions")
for s in ma_mt_sessions:
    print(f"  [{s['session_id']}] {s['description']} ({len(s['turns'])} turns)")
    for t in s["turns"]:
        tools = ", ".join(tc["name"] for tc in t["expected_tool_calls"]) or "—"
        print(f"    turn {t['turn_id']}: tools=[{tools}]")

#### Step 3: Run conversations and build per-turn payloads

Same shape as the booking example: one payload per turn, all sharing the same `session_id`. The adapter will group by `session_id` and produce one session dict per conversation — with `MultiAgentTrace`s in `per_turn_traces` and `full_trace` because the graph has multiple agent nodes.

In [ ]:
import time, uuid
from langchain_core.messages import HumanMessage
from uaef.adapters import LangGraphAdapter
from uaef.data.ground_truth import build_multi_turn_ground_truth

ma_mt_adapter = LangGraphAdapter()
ma_mt_turn_payloads = []     # flat list across all sessions
ma_mt_ground_truths = []
ma_mt_session_ids = []

for s in ma_mt_sessions:
    sid = s["session_id"]
    print(f"\n\u2500\u2500 Session {sid} ({len(s['turns'])} turns) \u2500\u2500")
    thread_id = str(uuid.uuid4())
    cfg = {"configurable": {"thread_id": thread_id}}

    # Track the events for this session so we can build context_documents
    # from tool results (used by hallucination_score).
    session_events = []

    for t in s["turns"]:
        query = str(t["query"])
        t_start = time.time()
        events = []
        for ev in multi_agent_graph_mt.stream(
            {"messages": [HumanMessage(content=query)]},
            cfg,
            stream_mode="updates",
        ):
            events.append(ev)
        lat = time.time() - t_start
        session_events.extend(events)
        print(f"  [{t['turn_id']}/{len(s['turns'])}] {query[:60]!r}  ({lat:.1f}s)")

        ma_mt_turn_payloads.append({
            "session_id": sid,
            "turn_id": t["turn_id"],
            "stream_events": [{"__human__": {"messages": [HumanMessage(content=query)]}}] + events,
            "latency": lat,
        })

    # build_multi_turn_ground_truth understands expected_tool_calls (parsed via
    # parse_ground_truth_row) and stores per-turn tools in
    # expected_arguments['per_turn_tools']. Pass session_events so tool results
    # become context_documents for hallucination scoring.
    ma_mt_ground_truths.append(
        build_multi_turn_ground_truth(s["turns"], events=session_events)
    )
    ma_mt_session_ids.append(sid)

print(f"\n\u2713 Collected {len(ma_mt_turn_payloads)} turn payloads across {len(ma_mt_session_ids)} sessions")
print(f"\nFirst session per-turn ground truth tools:")
_first_gt = ma_mt_ground_truths[0]
for i, tools in enumerate(_first_gt.expected_arguments.get("per_turn_tools", []), start=1):
    names = [tc.name for tc in tools] or ["—"]
    print(f"  turn {i}: {names}")

#### Step 4: Adapter → session dicts (with `MultiAgentTrace` per turn)

Single adapter call. The adapter sees more than one agent node in the events and produces `MultiAgentTrace` for both per-turn and full traces, while still returning the standard session-dict shape.

In [ ]:
ma_mt_traces = ma_mt_adapter.transform_to_canonical(ma_mt_turn_payloads)

print(f"✓ Built {len(ma_mt_traces)} session dicts")
for s in ma_mt_traces:
    full = s["full_trace"]
    full_type = type(full).__name__
    agents = list(full.agent_traces.keys()) if hasattr(full, "agent_traces") else []
    coord = len(full.coordination_events) if hasattr(full, "coordination_events") else 0
    per_turn_types = {type(t).__name__ for t in s["per_turn_traces"]}
    print(
        f"  [{s['session_id']}] full={full_type}  per_turn={per_turn_types}  "
        f"turns={len(s['per_turn_traces'])}  agents={agents}  coord_events={coord}"
    )

#### Step 5: Evaluate — multi-turn + multi-agent in a single call

`evaluate()` dispatches the session dict to `SingleAgentEvaluator.evaluate_multi_turn()`, which:

- runs per-turn-eligible metrics (tool calling, response quality, multi-agent, ...) on each `MultiAgentTrace` turn and averages them
- runs full-trace metrics (`context_retention`, `coherence`, ...) on the full `MultiAgentTrace`
- multi-agent metrics see the per-agent split + coordination events; multi-turn metrics see the aggregated `messages` (a computed property on `MultiAgentTrace`)

In [ ]:
from uaef.api import evaluate

ma_mt_metrics = single_ag_metrics + multi_ag_metrics

ma_mt_results = []
for sid, session_dict, gt in zip(ma_mt_session_ids, ma_mt_traces, ma_mt_ground_truths):
    res = evaluate(trace=session_dict, ground_truth=gt, metrics=ma_mt_metrics)
    ma_mt_results.append(res)
    print(f"  [{sid}] overall={res.overall_score:.2f}  passed={res.passed}  "
          f"is_multi_agent={res.metadata.get('is_multi_agent')}")

print(f"\n{'='*70}")
print("MULTI-TURN MULTI-AGENT EVALUATION")
print(f"{'='*70}")
for sid, res in zip(ma_mt_session_ids, ma_mt_results):
    print(f"\n\u2500\u2500 {sid} (overall={res.overall_score:.2f}) \u2500\u2500")
    for dim in res.dimension_results:
        print(f"  {dim.dimension_name} ({dim.aggregate_score:.2f}):")
        for m in dim.metric_scores:
            score = f"{m.score:.2f}" if m.score is not None else "N/A"
            print(f"    {m.metric_name:<30} {score}")

    # Per-turn metric table.
    # - Per-turn metrics (tool calling, response quality, multi-agent, ...) come
    #   from result.metadata['per_turn'][i]['metric_scores'].
    # - Full-trace metrics (multi_turn dimension) only have one value per
    #   conversation; we render them in a single 'Full' column.
    per_turn = res.metadata.get("per_turn") or []
    averages = res.metadata.get("per_turn_averages", {})
    if not per_turn:
        continue

    full_trace_only = {"context_retention", "coherence", "conversation_completeness", "turn_efficiency"}
    full_trace_scores = {}
    for dim in res.dimension_results:
        for m in dim.metric_scores:
            if m.metric_name in full_trace_only:
                full_trace_scores[m.metric_name] = m.score

    per_turn_metric_names = [
        m for m in per_turn[0]["metric_scores"].keys()
        if any(t["metric_scores"].get(m) is not None for t in per_turn)
    ]
    full_trace_metric_names = [m for m in full_trace_only if m in full_trace_scores]

    if not (per_turn_metric_names or full_trace_metric_names):
        continue

    col_w = 28
    print(f"\n  PER-TURN METRIC SCORES \u2014 {sid}")
    header = (
        f"  {'Metric':<{col_w}}"
        + "".join(f"{'Turn ' + str(t['turn']):>8}" for t in per_turn)
        + f"{'Avg':>9}"
    )
    print(header)
    print("  " + "-" * (len(header) - 2))

    for mname in per_turn_metric_names:
        row = f"  {mname:<{col_w}}"
        for t in per_turn:
            sc = t["metric_scores"].get(mname)
            row += f"{sc:>8.2f}" if sc is not None else f"{'N/A':>8}"
        avg = averages.get(mname)
        row += f"{avg:>9.2f}" if avg is not None else f"{'N/A':>9}"
        print(row)

    if full_trace_metric_names:
        # Full-trace metrics get N/A across the per-turn columns and their
        # actual score in the Avg column.
        print("  " + "-" * (len(header) - 2))
        for mname in full_trace_metric_names:
            row = f"  {mname:<{col_w}}"
            for _ in per_turn:
                row += f"{'\u2014':>8}"
            sc = full_trace_scores.get(mname)
            row += f"{sc:>9.2f}" if sc is not None else f"{'N/A':>9}"
            print(row)

In [ ]:
score = next(
    s for dim in res.dimension_results
    for s in dim.metric_scores
    if s.metric_name == "chain_of_thought_coherence"
)

# # Structured access — list of {turn, score, reasoning} dicts
# for entry in score.metadata["per_turn"]:
#     print(entry["turn"], entry["score"], entry["reasoning"])

# Pre-formatted multiline string suitable for direct printing
print(score.reasoning)

In [ ]:
for dim in res.dimension_results:
    for m in dim.metric_scores:
        if m.metric_name == "context_retention":
            print(f"[{sid}] context_retention reasoning: {m.reasoning}")


#### Step 6: Batch evaluation across sessions

`batch_evaluate()` accepts the list of session dicts and routes each through `evaluate_multi_turn()`.

In [ ]:
from uaef.api import batch_evaluate

ma_mt_batch = batch_evaluate(
    traces=ma_mt_traces,
    ground_truths=ma_mt_ground_truths,
    metrics=ma_mt_metrics,
    max_workers=2,
)

print(f"{'='*70}")
print(f"{'SESSION':<14}{'SCORE':>8}{'PASSED':>9}  DIMENSION SCORES")
print(f"{'='*70}")
for sid, r in zip(ma_mt_session_ids, ma_mt_batch):
    dims = "  ".join(f"{d.dimension_name[:8]}={d.aggregate_score:.2f}" for d in r.dimension_results)
    flag = "✓" if r.passed else "✗"
    print(f"{sid:<14}{r.overall_score:>8.2f}{flag:>9}  {dims}")
print(f"{'='*70}")

avg = sum(r.overall_score for r in ma_mt_batch) / len(ma_mt_batch)
pr = sum(1 for r in ma_mt_batch if r.passed) / len(ma_mt_batch) * 100
print(f"Average score: {avg:.2f}  |  Pass rate: {pr:.0f}%")

# Per-turn breakdown for the first session (mirrors the booking example)
first = ma_mt_batch[0]
per_turn = first.metadata.get("per_turn") or []
if per_turn:
    averages = first.metadata.get("per_turn_averages", {})
    metric_names = [
        m for m in per_turn[0]["metric_scores"].keys()
        if any(t["metric_scores"].get(m) is not None for t in per_turn)
    ]
    col_w = 26
    print(f"\n{'='*70}")
    print(f"PER-TURN METRIC SCORES — {ma_mt_session_ids[0]}")
    header = f"  {'Metric':<{col_w}}" + "".join(f"{'Turn ' + str(t['turn']):>8}" for t in per_turn) + f"{'Avg':>9}"
    print(header)
    print("  " + "-" * (len(header) - 2))
    for mname in metric_names:
        row = f"  {mname:<{col_w}}"
        for t in per_turn:
            sc = t["metric_scores"].get(mname)
            row += f"{sc:>8.2f}" if sc is not None else f"{'N/A':>8}"
        avg_v = averages.get(mname)
        row += f"{avg_v:>9.2f}" if avg_v is not None else f"{'N/A':>9}"
        print(row)

## Evaluate a Previously Deployed LangGraph Agent

If your LangGraph agent is already deployed (e.g. via LangServe, LangGraph Platform, or a custom HTTP endpoint), you can call it remotely and evaluate the responses.

**To test locally**, start the included test server in a separate terminal:
```bash
python serve_langgraph_agent.py
```
Then run the cells below with `AGENT_ENDPOINT = "http://localhost:8123/invoke"`.

#### Configure the deployed agent endpoint

In [ ]:
# Point to your deployed agent (or the local test server)
AGENT_ENDPOINT = "http://localhost:8123/invoke"

# Node names must match the graph that the deployed agent uses
AGENT_NODE_NAME = "agent"
TOOL_NODE_NAME = "tools"

print(f"Agent endpoint: {AGENT_ENDPOINT}")
print(f"Agent node: {AGENT_NODE_NAME}, Tool node: {TOOL_NODE_NAME}")

#### Helper: call the deployed agent and build an AgentTrace

This cell has **no LangChain dependency**. It converts the JSON response into lightweight objects that the LangGraphAdapter can process via `getattr()`.

In [ ]:
import requests, time as _time
from langchain_core.messages import AIMessage, HumanMessage, ToolMessage
from uaef.adapters import LangGraphAdapter

adapter = LangGraphAdapter()

def _dict_to_msg(d):
    """Convert a serialized message dict to the appropriate LangChain message type.

    Handles both LangChain internal type strings ("ai", "human", "tool") and
    class-name strings returned by some servers ("AIMessage", "HumanMessage", "ToolMessage").
    """
    msg_type = d.get("type", "").lower()
    content = d.get("content", "")
    tool_calls = d.get("tool_calls", [])
    usage_metadata = d.get("usage_metadata")

    if msg_type in ("human", "humanmessage"):
        return HumanMessage(content=content)
    elif msg_type in ("tool", "toolmessage") or d.get("tool_call_id"):
        return ToolMessage(content=content, tool_call_id=d.get("tool_call_id", ""))
    else:
        # "ai", "aimessage", or unknown — treat as AIMessage
        msg = AIMessage(content=content, tool_calls=tool_calls)
        if usage_metadata:
            msg.usage_metadata = usage_metadata
        return msg

def _hydrate_events(events):
    """Convert JSON events into the format the LangGraphAdapter expects."""
    hydrated = []
    for evt in events:
        h_evt = {}
        for node_name, node_data in evt.items():
            if isinstance(node_data, dict) and "messages" in node_data:
                h_evt[node_name] = {
                    "messages": [_dict_to_msg(m) for m in node_data["messages"]]
                }
            else:
                h_evt[node_name] = node_data
        hydrated.append(h_evt)
    return hydrated

def invoke_deployed_agent(query, session_id="default"):
    """Call the deployed agent endpoint and return an AgentTrace."""
    t0 = _time.time()
    try:
        resp = requests.post(AGENT_ENDPOINT, json={"input": query, "session_id": session_id}, timeout=60)
        latency = _time.time() - t0
        resp.raise_for_status()
        raw_events = resp.json()["events"]
    except requests.exceptions.RequestException as e:
        raise RuntimeError(f"Failed to invoke agent at {AGENT_ENDPOINT}: {e}") from e

    # Prepend the human message so the adapter picks it up as MessageRole.USER
    events = [{"__human__": {"messages": [HumanMessage(content=query)]}}] + _hydrate_events(raw_events)
    return adapter.transform_to_canonical({
        "stream_events": events,
        "session_id": session_id,
        "tool_node_name": TOOL_NODE_NAME,
        "latency": latency,
    })

print("✓ invoke_deployed_agent() ready")

#### Single evaluation against the deployed agent

In [ ]:
from uaef.api import evaluate
from uaef.models import GroundTruth, ToolCall
from datetime import datetime, timezone

agent_trace = invoke_deployed_agent("What's the weather in Seattle?")

print(f"✓ Got response from deployed agent")
print(f"  Messages: {len(agent_trace.messages)}")
print(f"  Tool Calls: {len(agent_trace.tool_calls)}")
if agent_trace.latency:
    print(f"  Latency: {agent_trace.latency:.3f}s")
for msg in agent_trace.messages:
    print(f"  [{msg.role.value}]: {msg.content[:200]}")

ground_truth = GroundTruth(
    expected_output="The weather in Seattle is 65°F and cloudy",
    expected_tool_calls=[
        ToolCall(name="get_weather", arguments={"location": "Seattle"}, timestamp=datetime.now(timezone.utc))
    ],
)

result = evaluate(trace=agent_trace, ground_truth=ground_truth, metrics=metrics)

print(f"\n{'='*50}")
print("DEPLOYED LANGGRAPH AGENT — EVALUATION RESULTS")
print(f"{'='*50}")
print(f"Overall Score: {result.overall_score:.2f}")
print(f"Passed: {'✓ Yes' if result.passed else '✗ No'}")
for dimension in result.dimension_results:
    print(f"\n{dimension.dimension_name} ({dimension.aggregate_score:.2f}):")
    for metric in dimension.metric_scores:
        score_str = f"{metric.score:.2f}" if metric.score is not None else "N/A"
        print(f"  {metric.metric_name}: {score_str}")
        if metric.score is None:
            print(f"    reason: {metric.reasoning}")

#### Batch evaluation against the deployed agent

In [ ]:
import pandas as pd
import json
from uaef.api import batch_evaluate
from uaef.models.message import MessageRole

excel_path = "data/ground-truth.xlsx"
df = pd.read_excel(excel_path)
gt_json = df.to_dict(orient="records")
print(f"✓ Loaded {len(gt_json)} ground truth entries")

traces = []
ground_truths = []

for i, row in enumerate(gt_json):
    query = str(row.get("query", row.get("input", row.get("Question", ""))))
    expected = str(row.get("expected_output", row.get("expected", row.get("Answer", ""))))
    context = str(row.get("context", "")) if pd.notna(row.get("context")) else ""

    expected_tools = []
    raw_tools = row.get("expected_tool_calls", row.get("tools", None))
    if pd.notna(raw_tools) and raw_tools:
        try:
            parsed = json.loads(str(raw_tools)) if isinstance(raw_tools, str) else raw_tools
            if isinstance(parsed, list):
                for t in parsed:
                    expected_tools.append(ToolCall(
                        name=t.get("name", t.get("tool_name", "")),
                        arguments=t.get("arguments", t.get("parameters", {})),
                        timestamp=datetime.now(timezone.utc),
                    ))
        except (json.JSONDecodeError, TypeError):
            pass

    trace = invoke_deployed_agent(query, session_id=f"batch_{i}")
    traces.append(trace)
    ground_truths.append(GroundTruth(
        expected_output=expected,
        expected_tool_calls=expected_tools,
        context_documents=[context] if context else [],
    ))

    label = f"'{query[:50]}...'" if len(query) > 50 else f"'{query}'"
    print(f"  [{i+1}/{len(gt_json)}] {label}")

print(f"\n✓ Ran {len(traces)} queries through the deployed agent")

batch_results = batch_evaluate(traces=traces, ground_truths=ground_truths, metrics=metrics, max_workers=4)

print(f"\n{'='*50}")
print(f"BATCH RESULTS — DEPLOYED LANGGRAPH AGENT")
print(f"{'='*50}")
for i, r in enumerate(batch_results):
    status = "✓" if r.passed else "✗"
    print(f"  {status} Test {i+1}: {r.overall_score:.2f}")

avg = sum(r.overall_score for r in batch_results) / len(batch_results)
pr = sum(1 for r in batch_results if r.passed) / len(batch_results) * 100
print(f"\nAverage Score: {avg:.2f}")
print(f"Pass Rate: {pr:.1f}%")

In [ ]:
# Uncomment to print evaluation details

# for i, result in enumerate(batch_results):
#     print(f"test {i}:")
#     for dimension in result.dimension_results:
#         print(f"\n{dimension.dimension_name} (aggregate score: {dimension.aggregate_score:.2f}):")
#         for metric in dimension.metric_scores:
#             if metric.score is None:
#                 print(f"  {metric.metric_name}: {metric.score}")
#                 print(metric.reasoning)
#             else:    
#                 print(f"  {metric.metric_name}: {metric.score:.2f}")
            
#     print('-'*30)

#### Export deployed agent evaluation results

In [ ]:
# Save evaluation results
from uaef.utils import save_metric_results

filepath = save_metric_results(batch_results, gt_json, prefix="langgraph_deployed_batch_results")


## Integrating Langfuse 

# Cell: Configure Langfuse + run agent with callback handler
from langfuse.callback import CallbackHandler as LangfuseCallbackHandler
from langchain_core.messages import HumanMessage
from datetime import datetime, timezone

# Initialize Langfuse callback handler
# Set LANGFUSE_PUBLIC_KEY, LANGFUSE_SECRET_KEY, LANGFUSE_HOST in your .env
langfuse_handler = LangfuseCallbackHandler(
    trace_name="langgraph-weather-eval",
    session_id="lg_langfuse_001",
    tags=["evaluation", "langgraph"],
)

# Run the LangGraph agent with Langfuse tracing
user_input = "What's the weather in Seattle?"
events = []
for event in graph.stream(
    {"messages": [HumanMessage(content=user_input)]},
    stream_mode="updates",
    config={"callbacks": [langfuse_handler]},
):
    events.append(event)

# Flush to make sure all data is sent to Langfuse
langfuse_handler.langfuse.flush()

# Get the trace ID that Langfuse assigned
langfuse_trace_id = langfuse_handler.get_trace_id()
print(f"✓ Agent ran with Langfuse tracing")
print(f"  Trace ID: {langfuse_trace_id}")

# Print the agent's final response
final_messages = [e for e in events if "agent" in e]
if final_messages:
    print(f"  Response: {final_messages[-1]['agent']['messages'][-1].content}")


In [ ]:
# Cell: Fetch the trace from Langfuse and transform with LangfuseAdapter
from langfuse import Langfuse
from uaef.adapters import LangfuseAdapter
from uaef.api import evaluate
from uaef.models import GroundTruth, ToolCall

# Fetch the full trace from Langfuse
langfuse_client = Langfuse()
trace = langfuse_client.fetch_trace(langfuse_trace_id)

# Build the raw_data dict the adapter expects from the Langfuse trace
# trace.data contains the trace object, trace.data.observations has the observation list
trace_data = trace.data

langfuse_raw = {
    "trace_id": trace_data.id,
    "session_id": trace_data.session_id,
    "input": trace_data.input,
    "output": trace_data.output,
    "metadata": trace_data.metadata or {},
    "timestamp": trace_data.timestamp.isoformat() if trace_data.timestamp else None,
    "observations": [],
}

# Fetch observations for this trace
observations = langfuse_client.fetch_observations(trace_id=langfuse_trace_id)
for obs in observations.data:
    langfuse_raw["observations"].append({
        "id": obs.id,
        "trace_id": obs.trace_id,
        "type": obs.type,  # "generation", "span", or "event"
        "name": obs.name,
        "input": obs.input,
        "output": obs.output,
        "metadata": obs.metadata,
        "usage": {
            "input": obs.usage.input if obs.usage else 0,
            "output": obs.usage.output if obs.usage else 0,
            "total": obs.usage.total if obs.usage else 0,
        } if obs.usage else {},
        "model": obs.model,
        "start_time": obs.start_time.isoformat() if obs.start_time else None,
        "end_time": obs.end_time.isoformat() if obs.end_time else None,
        "parent_observation_id": obs.parent_observation_id,
    })

# Transform to UAEF canonical format
adapter = LangfuseAdapter()
agent_trace = adapter.transform_to_canonical(langfuse_raw)

print(f"✓ Transformed Langfuse trace to AgentTrace")
print(f"  Trace ID: {agent_trace.trace_id}")
print(f"  Messages: {len(agent_trace.messages)}")
print(f"  Tool Calls: {len(agent_trace.tool_calls)}")
print(f"  Framework: {agent_trace.framework}")
if agent_trace.input_tokens:
    print(f"  Input Tokens: {agent_trace.input_tokens}")
if agent_trace.output_tokens:
    print(f"  Output Tokens: {agent_trace.output_tokens}")
if agent_trace.latency:
    print(f"  Latency: {agent_trace.latency:.3f}s")

for msg in agent_trace.messages:
    print(f"\n  [{msg.role.value}]: {msg.content[:200]}")


In [ ]:
# Cell: Evaluate
ground_truth = GroundTruth(
    expected_output="The weather in Seattle is 65°F and cloudy",
    expected_tool_calls=[
        ToolCall(
            name="get_weather",
            arguments={"location": "Seattle"},
            timestamp=datetime.now(timezone.utc),
        )
    ],
    context_documents=["Seattle is a city in Washington state"],
)

result = evaluate(
    trace=agent_trace,
    ground_truth=ground_truth,
    metrics=metrics,
)

print(f"\n{'='*50}")
print("LANGGRAPH (via LANGFUSE) - EVALUATION RESULTS")
print(f"{'='*50}")
print(f"Overall Score: {result.overall_score:.2f}")
print(f"Passed: {'✓ Yes' if result.passed else '✗ No'}")

print(f"\n{'='*50}")
print("METRIC SCORES BY DIMENSION")
for dimension in result.dimension_results:
    print(f"\n{dimension.dimension_name} (aggregate score: {dimension.aggregate_score:.2f}):")
    for metric in dimension.metric_scores:
        print(f"  {metric.metric_name}: {metric.score:.2f}")


In [ ]:
## batch evaluation on langgraph + lagnfuse adapters